In [1]:
import pandas as pd
import numpy as np
import duckdb
import lightgbm as lgb
from datetime import datetime
from pandas.tseries.offsets import MonthEnd
import gc
import glob
import os

In [2]:
def gather_data(start_date_1,end_date_1,start_date_2=None,end_date_2=None):
    if start_date_2==None:
        start_date_2=start_date_1
    if end_date_2==None:
        end_date_2=end_date_1

    #query the features parquet for rows with desired dates
    query = """
    SELECT *
    FROM 'features.parquet'
    WHERE (
        (date >= ? AND date <= ?)
        OR
        (date >= ? AND date <= ?)
    )
    ORDER BY date, he
    """

    return con.execute(
        query,
        [start_date_1, end_date_1, start_date_2, end_date_2]
    ).df()

In [3]:
def train_model(train_data,feature_cols):

    #prepare training data with intended feature_columns
    X_train = train_data[feature_cols]

    # designate categorical features
    cols=X_train.columns
    if "bus_unique_id" in cols:
        X_train["bus_unique_id"] = X_train["bus_unique_id"].astype("category")
    if "zone_name" in cols:
        X_train["zone_name"] = X_train["zone_name"].astype("category")

    #prepare actual values for training
    y_train = train_data["pd"]

    #train the model on the data
    model = lgb.LGBMRegressor(force_row_wise=True,verbosity=-1)
    trained_model = model.fit(X_train,y_train)
    
    return trained_model

In [4]:
def predict_demand(model,prediction_day_data,feature_cols,model_name,predicted_on):
    
    #prepare future data with intended feature_columns
    X_test=prediction_day_data[feature_cols]

    # designate categorical features
    cols=X_test.columns
    if "bus_unique_id" in cols:
        X_test["bus_unique_id"] = X_test["bus_unique_id"].astype("category")
    if "zone_name" in cols:
        X_test["zone_name"] = X_test["zone_name"].astype("category")
    
    #get predictions for future data
    results=model.predict(X_test)
    
    return pd.DataFrame({
        "model_name" : model_name,
        "forecast_created_at" : predicted_on,
        "target_date" : prediction_day_data["date"],
        "he" : prediction_day_data["he"],
        "bus_id" : prediction_day_data["bus_unique_id"],
        "zone_id" : prediction_day_data["zone_name"],
        "predict_pd" : results,
        "actual_pd":prediction_day_data["pd"],
        "baseline_pd":prediction_day_data["same_hour_prev_year"]
    })

In [5]:
def calculate_statistics(df,month):

        #calculate statistics on predictions and baseline
        MAE_predict =(df["actual_pd"] - df["predict_pd"]).abs().mean()
        RMSE_predict = (((df["actual_pd"] - df["predict_pd"]) ** 2).mean()) ** 0.5
        WMAPE_predict = ((df["actual_pd"] - df["predict_pd"]).abs().sum() / df["actual_pd"].sum())
        MAE_baseline =(df["actual_pd"] - df["baseline_pd"]).abs().mean()
        RMSE_baseline = (((df["actual_pd"] - df["baseline_pd"]) ** 2).mean()) ** 0.5
        WMAPE_baseline = ((df["actual_pd"] - df["baseline_pd"]).abs().sum() / df["actual_pd"].sum())
        statistics={"month":month,
                    "MAE_predict":MAE_predict,
                    "RMSE_predict":RMSE_predict,
                    "WMAPE_predict":WMAPE_predict,
                    "MAE_baseline":MAE_baseline,
                    "RMSE_baseline":RMSE_baseline,
                    "WMAPE_baseline":WMAPE_baseline
        }
        return statistics


In [6]:
con=duckdb.connect()

In [7]:
dates = pd.date_range(
    start="2022-01-01",
    end="2025-12-31",
    freq="D"
)
first_day_2025_idx=dates.get_loc('2025-01-01')

In [8]:
#designate the feature columns to be used for the next day model
next_day_feature_cols = [
"bus_unique_id",

"he",
"dow",
"month",
"day_of_year",
"is_weekend",

"same_hour_prev_week",
"same_hour_prev_35day",
"same_hour_prev_year",
    
]

#remove features that may be unknown at the time of training the next month model
monthly_feature_cols = next_day_feature_cols
monthly_feature_cols.remove("same_hour_prev_week")

In [9]:
#loop initialization variables
daily_retrain_frequency=7
train_data_lags={"start_1" : 37,
                 "end_1" : 2,
}
monthly_train_data_lags={"start_1" : 37,
                         "end_1" : 2,
}

In [10]:
days_until_retrain=0 #reset the retraining countdown to train the next_day model on the first day

#initialize a dataframe for the required statistics
next_day_statistics=pd.DataFrame(columns=["month","MAE_predict","RMSE_predict","WMAPE_predict","MAE_baseline","RMSE_baseline","WMAPE_baseline"])
next_month_statistics=pd.DataFrame(columns=["month","MAE_predict","RMSE_predict","WMAPE_predict","MAE_baseline","RMSE_baseline","WMAPE_baseline"])


#Loop through every day of the test period --------------------------------------------------------------------------------------------------------------------------------------------------------------------
for i in range(first_day_2025_idx,len(dates)):
    
    #get time based variables
    prediction_day=dates[i]
    previous_day=dates[i-1]
    week=prediction_day.week 
    month=prediction_day.month 
    year=prediction_day.year
    first_day_month = pd.Timestamp(year=year, month=month, day=1)
    last_day_month = first_day_month + MonthEnd(0)

    
#Gather future data for the whole month on the first day of the month------------------------------------------------------------------------------------------------------------------------------------------
    if prediction_day==first_day_month:
        future_data=gather_data(first_day_month,last_day_month)


#Tain monthly model----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------    
    if prediction_day==first_day_month:
        print(f"\rProcessing month {month}/12", end="")
        
        train_data=gather_data(dates[i-monthly_train_data_lags["start_1"]],dates[i-monthly_train_data_lags["end_1"]]) #gather the training data

        next_month_model=train_model(train_data,monthly_feature_cols) #train the next month forcast model
        
        monthly_predicted_on=previous_day #save the last day of the last month as the forcast date
        
        del train_data #delete train_data variable
        gc.collect

        #reset the monthly and daily model predictions for the new month
        monthly_predictions=[]
        next_day_predictions=[]

#Tain daily model--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------        
    if days_until_retrain==0:
        days_until_retrain=daily_retrain_frequency#reset the retrain countdown
        
        train_data=gather_data(dates[i-train_data_lags["start_1"]],dates[i-train_data_lags["end_1"]])#gather the training data
        
        next_day_model=train_model(train_data,next_day_feature_cols)#train the next day forcast model
        
        del train_data #delete train_data variable
        gc.collect
        
    days_until_retrain-=1 #decrease the retrain countdown for the next day model
    
        
#Make Predictions for prediction_day----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------       
    prediction_day_data=future_data[future_data["date"]==prediction_day] #get the data for the prediction day
    
    if not prediction_day_data.empty:
        #get predictions from both models 
        next_day_model_predictions=predict_demand(next_day_model,prediction_day_data,next_day_feature_cols,'next-day model',previous_day)
        monthly_model_predictions=predict_demand(next_month_model,prediction_day_data,monthly_feature_cols,'next-month model',monthly_predicted_on)

        #append the predictions to this months list of predictions
        next_day_predictions.append(next_day_model_predictions)
        monthly_predictions.append(monthly_model_predictions)

#Export predictions to a parquet file at the end of every month---------------------------------------------------------------------------------------------------------------------------------------------------------       
    if prediction_day==last_day_month:
        daily_df=pd.concat(next_day_predictions,ignore_index=True) #concatenate the predictions for the month
        daily_df["predict_pd"] = daily_df["predict_pd"].clip(lower=0) #correct negative demand predictions
        
        daily_df.to_parquet(f"temporary_bus_daily_predictions_{year}_{month:02d}.parquet",engine="pyarrow",index=False) #export predictions to a parquet
        
        del next_day_predictions #delete predictions list for the month
        gc.collect
        
        monthly_df=pd.concat(monthly_predictions,ignore_index=True) #concatenate the predictions for the month
        monthly_df["predict_pd"] = monthly_df["predict_pd"].clip(lower=0) #correct negative demand predictions
        
        monthly_df.to_parquet(f"temporary_bus_monthly_predictions_{year}_{month:02d}.parquet",engine="pyarrow",index=False) #export predictions to a parquet
        
        del monthly_predictions #delete predictions list for the month
        gc.collect

#Append statistics for the month to the respective statistics dataframe---------------------------------------------------------------------------------------------------------------------------------------------
        next_day_statistics.loc[len(next_day_statistics)]=calculate_statistics(daily_df,month)
        next_month_statistics.loc[len(next_month_statistics)]=calculate_statistics(monthly_df,month)
        
print(f"\rDone                  ", end="")

Done                  

In [11]:
#combine all prediction parquets into a single file
try:
    con.execute("""
        
            COPY (
            
            SELECT * EXCLUDE (actual_pd, baseline_pd)
            FROM 'temporary_bus_daily_predictions*.parquet'
            UNION ALL
            SELECT * EXCLUDE (actual_pd, baseline_pd)
            FROM 'temporary_bus_monthly_predictions*.parquet'
            )
            
            TO 'direct_bus_predictions.parquet'
            (FORMAT PARQUET)
            
            """)
            
    #remove temporary prediction files        
    for file in glob.glob("temporary_bus_daily_predictions*.parquet"):
         os.remove(file)
    for file in glob.glob("temporary_bus_monthly_predictions*.parquet"):
        os.remove(file)
except duckdb.IOException:
    print("predictions can not be combined")

In [16]:
next_day_statistics

,month,MAE_predict,RMSE_predict,WMAPE_predict,MAE_baseline,RMSE_baseline,WMAPE_baseline
0,1,2.387764,6.862588,0.210059,3.831457,11.061389,0.317284
1,2,1.997500,5.641275,0.186147,3.554443,11.946355,0.312030
2,3,1.733506,6.757604,0.177911,2.911822,10.161469,0.280035
3,4,1.758620,4.302648,0.167017,3.188050,12.441107,0.282010
4,5,1.685194,4.957673,0.147046,3.557385,13.442819,0.291735
5,6,2.255285,6.072237,0.173790,3.531206,12.467493,0.256965
6,7,1.826088,4.848813,0.137657,3.754201,12.995378,0.267143
7,8,1.764010,4.844982,0.128654,3.425099,13.085479,0.236132
8,9,1.051570,3.384064,0.083521,3.110914,12.320311,0.232261
9,10,1.391907,3.799051,0.123172,3.104611,10.317633,0.256373


In [17]:
next_month_statistics

,month,MAE_predict,RMSE_predict,WMAPE_predict,MAE_baseline,RMSE_baseline,WMAPE_baseline
0,1,3.072982,10.685086,0.270340,3.831457,11.061389,0.317284
1,2,2.091957,6.007758,0.194949,3.554443,11.946355,0.312030
2,3,2.091702,7.331263,0.214673,2.911822,10.161469,0.280035
3,4,2.104668,4.907865,0.199881,3.188050,12.441107,0.282010
4,5,1.846672,5.154164,0.161136,3.557385,13.442819,0.291735
5,6,3.042145,8.703941,0.234424,3.531206,12.467493,0.256965
6,7,2.056123,5.734839,0.154998,3.754201,12.995378,0.267143
7,8,1.897869,6.037065,0.138417,3.425099,13.085479,0.236132
8,9,1.274002,3.904094,0.101188,3.110914,12.320311,0.232261
9,10,1.455252,4.230499,0.128778,3.104611,10.317633,0.256373
